# 🌍 Never-AI — Super-Trainer GPU « Modèle Mondial Cumulatif · X & Réseaux » (v3.1)

Ce notebook fait grandir **le vrai modèle partagé `Never-AI` (`ChronoMorph-LM Sprout-MoE`)** hébergé sur GitHub — **pas seulement votre ordinateur** :

1. **⬇️ Reprise Cumulative (`Cellule 4`)** : télécharge d'abord le **dernier cerveau mondial `Never-AI`** depuis GitHub Releases pour **continuer son apprentissage là où le précédent s'est arrêté** (aucune remise à zéro !).
2. **𝕏 Entraînement Libre Sans Mode ni Directive (`Cellule 3 & 4`)** : absorbe des milliers de vrais posts **X / Twitter**, fils de réponses en direct et discussions humaines francophones. Son nom reste **Never-AI**, et sa personnalité émerge 100 % de ses propres poids.
3. **⬆️ Contribution Mondiale Directe (`Cellule 6`)** : fusionne votre entraînement (`Federated WSM`) et publie directement le nouveau cerveau sur GitHub (`mouusieur/never-ai`) depuis les serveurs de Google (**`0 Mo` consommés sur votre propre connexion** !).

In [ ]:
# 1️⃣ VÉRIFICATION GPU & COFFRE COLAB (ZÉRO BLOCAGE GOOGLE DRIVE !)
import os
import torch

TRY_GOOGLE_DRIVE = False
VAULT_DIR = '/content/NeverAI_Vault/checkpoints'
DRIVE_BACKUP_DIR = None

if TRY_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_BACKUP_DIR = '/content/drive/MyDrive/NeverAI_Vault/checkpoints'
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    except Exception as err:
        print(f'ℹ️ Google Drive ignoré ({err}) -> SSD local Colab utilisé.')

os.makedirs(VAULT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_label = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU (Activez T4 GPU dans Exécution -> Modifier le type d\'exécution !)'
print(f'✅ Accélérateur actif : {gpu_label} | PyTorch {torch.__version__}')

In [ ]:
# 2️⃣ CLONAGE NEVER-AI & INSTALLATION ULTRA-RAPIDE (~8 SECONDES)
import os, subprocess, sys

if not os.path.exists('/content/never-ai/.git'):
    subprocess.run([
        'git', 'clone', '-b', 'arena/403f5d7f-never-ai', '--depth', '1',
        'https://github.com/mouusieur/never-ai.git', '/content/never-ai'
    ], check=True)
else:
    subprocess.run(['git', '-C', '/content/never-ai', 'pull', '--ff-only'], check=False)

os.chdir('/content/never-ai')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tokenizers', 'safetensors', 'sympy', 'pyyaml', 'tqdm', 'datasets'
], check=True)

if '/content/never-ai/src' not in sys.path:
    sys.path.insert(0, '/content/never-ai/src')
print('✅ Moteur Never-AI v3.1 (Cerveau Mondial Cumulatif) prêt !')

In [ ]:
# 3️⃣ COLLECTE BRUTE SUR X / TWITTER & LES RÉSEAUX SOCIAUX (ZÉRO MODE, ZÉRO DIRECTIVE)
import re
from datasets import load_dataset
from never_ai.core.special_tokens import BOS_TOKEN, EOS_TOKEN, NEVER_TOKEN, USER_TOKEN
from never_ai.data.social_stream import SocialStreamHarvester

raw_texts_for_tokenizer = []
formatted_docs = []

# Ancrage factuel minimal de son propre nom : Never-AI (aucune règle de ton ou d'opinion !)
for q_id, a_id in [
    ('tu t\'appelles comment ?', 'Never-AI.'),
    ('c\'est quoi ton nom ?', 'Moi c\'est Never-AI.'),
    ('qui es-tu ?', 'Never-AI.'),
] * 25:
    formatted_docs.append(f'{BOS_TOKEN}{USER_TOKEN}{q_id}{NEVER_TOKEN}{a_id}{EOS_TOKEN}')

def clean_post(t: str) -> str:
    t = re.sub(r'https?://\S+', '', t or '').strip()
    t = re.sub(r'(?:@user\s*)+', '', t, flags=re.I).strip()
    return t

# A) Vrais tweets X / Twitter francophones bruts (étiquettes ignorées)
print('𝕏 Chargement de vrais posts X / Twitter francophones bruts...')
x_tweets = []
for ds_name, cfg_name in [
    ('cardiffnlp/tweet_sentiment_multilingual', 'french'),
    ('mteb/tweet_sentiment_multilingual', 'french'),
]:
    try:
        for split_name in ('train', 'validation', 'test'):
            ds_x = load_dataset(ds_name, cfg_name, split=split_name, streaming=True)
            for row in ds_x:
                tw = clean_post(row.get('text') or '')
                if len(tw) >= 12:
                    x_tweets.append(tw)
                    raw_texts_for_tokenizer.append(tw)
                    formatted_docs.append(f'{BOS_TOKEN}{tw}{EOS_TOKEN}')
            if len(x_tweets) >= 8000:
                break
        if len(x_tweets) >= 3000:
            break
    except Exception as e:
        print(f'   -> Info {ds_name}: {e}')
print(f'   -> {len(x_tweets):,} vrais tweets X francophones bruts chargés !')

# B) Flux réseaux sociaux & réponses en direct (X miroirs + Microblog FR + Reddit FR)
print('📡 Aspiration en direct des posts et réponses sur les réseaux...')
harvester = SocialStreamHarvester()
live_bundle = harvester.harvest_live_bundle(x_query='lang:fr', max_per_source=120)
formatted_docs.extend(live_bundle['sequences'])
raw_texts_for_tokenizer.extend(live_bundle['samples'])
print('   -> Flux live capturé :', live_bundle['counts'])

# C) Avis & réactions spontanées d'internautes francophones
print('🔥 Streaming de réactions spontanées d\'internautes francophones...')
opinions_n = 0
try:
    ds_op = load_dataset('tblard/allocine', split='train', streaming=True)
    for row in ds_op:
        rev = clean_post(row.get('review') or '')
        if 25 <= len(rev) <= 900:
            raw_texts_for_tokenizer.append(rev)
            formatted_docs.append(f'{BOS_TOKEN}{rev}{EOS_TOKEN}')
            opinions_n += 1
            if opinions_n >= 5000:
                break
    print(f'   -> {opinions_n:,} réactions spontanées chargées !')
except Exception as e:
    print(f'   -> Info Allociné ({e})')

# D) Fils de discussions humains bruts (OpenAssistant FR)
print('💬 Streaming de fils de discussion humains francophones...')
dialog_n = 0
try:
    ds_oasst = load_dataset('OpenAssistant/oasst2', split='train', streaming=True)
    by_id = {}
    for row in ds_oasst:
        if row.get('lang') != 'fr':
            continue
        mid, pid = row.get('message_id'), row.get('parent_id')
        txt = (row.get('text') or '').strip()
        if not txt:
            continue
        by_id[mid] = txt
        if pid in by_id:
            parent_txt = by_id[pid]
            raw_texts_for_tokenizer.append(parent_txt + '\n' + txt)
            formatted_docs.append(f'{BOS_TOKEN}{USER_TOKEN}{parent_txt}{NEVER_TOKEN}{txt}{EOS_TOKEN}')
            dialog_n += 1
            if dialog_n >= 5000:
                break
except Exception as e:
    print(f'   -> Info OASST2 ({e})')
print(f'   -> {dialog_n:,} échanges directs chargés !')

# E) Culture générale (Wikipedia FR)
print('📚 Streaming de 4 000 articles Wikipedia FR...')
ds_wiki = load_dataset('wikimedia/wikipedia', '20231101.fr', split='train', streaming=True)
wiki_n = 0
for row in ds_wiki:
    txt = (row.get('text') or '').strip()
    if len(txt) > 400:
        excerpt = txt[:2800]
        raw_texts_for_tokenizer.append(excerpt)
        formatted_docs.append(f'{BOS_TOKEN}{excerpt}{EOS_TOKEN}')
        wiki_n += 1
        if wiki_n >= 4000:
            break

total_chars = sum(len(t) for t in formatted_docs)
print(f'✅ Corpus prêt : {len(formatted_docs):,} séquences ({total_chars / 1e6:.1f} Mo)')

In [ ]:
# 4️⃣ REPRISE DEPUIS LE CERVEAU MONDIAL GITHUB + ENTRAÎNEMENT GPU CUMULATIF !
from pathlib import Path
import random, shutil, tempfile, time
from safetensors.torch import load_file
import torch
from never_ai.core.config import ChronoMorphConfig
from never_ai.core.model import ChronoMorphLM
from never_ai.core.tokenizer import MorphTokenizer
from never_ai.data.curriculum import ContinualDatasetAndReplay
from never_ai.optim.composite_muon import CompositeMuonOptimizer, WSMCheckpointMerger
from never_ai.vault.global_brain_sync import GlobalBrainSync, transfer_or_merge_weights
from never_ai.vault.manager import VaultManager

STEPS = 2000
BATCH_SIZE = 12 if device == 'cuda' else 2
SEQ_LEN = 384 if device == 'cuda' else 128

sync = GlobalBrainSync()
tok = None
remote_sd = None
remote_tok = None

print('🌍 Téléchargement du cerveau mondial actuel Never-AI depuis GitHub Releases pour continuer dessus...')
try:
    tmp_glob = Path(tempfile.mkdtemp())
    dl_info = sync.download_global_brain_to_dir(tmp_glob)
    st_path = tmp_glob / ('model_wsm.safetensors' if (tmp_glob / 'model_wsm.safetensors').exists() else 'model_live.safetensors')
    remote_sd = load_file(str(st_path), device='cpu')
    if (tmp_glob / 'tokenizer.json').exists():
        remote_tok = MorphTokenizer.load(tmp_glob / 'tokenizer.json')
        if remote_tok.vocab_size >= 8000:
            tok = remote_tok
            print(f'   -> Tokenizer mondial conservé (vocab={tok.vocab_size}) !')
    print(f"   -> Cerveau mondial récupéré ({dl_info['bytes']/1048576:.1f} Mo) : reprise cumulative activée !")
except Exception as e:
    print(f'   -> Premier cerveau 8K ou réseau indisponible ({e}) : initialisation fraîche.')

if tok is None:
    print('🔤 Entraînement du tokenizer BPE (vocab=8192) sur le français des réseaux...')
    tok = MorphTokenizer.train_from_iterator(raw_texts_for_tokenizer, vocab_size=8192)

cfg = ChronoMorphConfig.sprout_moe(vocab_size=tok.vocab_size)
model = ChronoMorphLM(cfg).to(device)
if remote_sd is not None:
    tr_stats = transfer_or_merge_weights(model, tok, remote_sd, source_tok=remote_tok, blend_alpha=1.0)
    print('🧬 Poids du cerveau mondial injectés avant entraînement :', tr_stats)

counts = model.count_parameters()
print(f"🧠 Never-AI ({cfg.stage_name}) sur {device.upper()} : {counts['total_params']/1e6:.2f}M paramètres")

rng = random.Random()
rng.shuffle(formatted_docs)
split_idx = max(16, int(len(formatted_docs) * 0.97))
dataset = ContinualDatasetAndReplay(tok, seq_len=SEQ_LEN)
stats_ds = dataset.load_corpora(formatted_docs[:split_idx], formatted_docs[split_idx:])
print(f"📦 Séquences prêtes : {stats_ds['train_chunks']:,} train | {stats_ds['val_chunks']:,} val")

opt = CompositeMuonOptimizer(model, muon_lr=2.5e-3, adamw_lr=7.0e-4, weight_decay=0.01, warmup_steps=80)
wsm = WSMCheckpointMerger(max_snapshots=6)
vault = VaultManager(vault_root=VAULT_DIR)
amp_dtype = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float32

t0 = time.time()
tokens_seen = 0
model.train()
for step in range(1, STEPS + 1):
    batch = dataset.sample_batch(batch_size=BATCH_SIZE, split='train', replay_ratio=0.20, device=device)
    opt.zero_grad()
    with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(amp_dtype != torch.float32)):
        out = model(batch.input_ids, targets=batch.targets, loss_mask=batch.loss_mask, update_titans=True)
        loss = out.loss
    loss.backward()
    opt.step(model)
    model.update_moe_biases()
    tokens_seen += int(batch.input_ids.numel())

    if step % 100 == 0 or step == 1:
        dt = max(1e-3, time.time() - t0)
        print(f'⚡ Step {step:4d}/{STEPS} | CE Loss = {out.ce_loss:.4f} | {tokens_seen / dt:,.0f} tok/s | +{tokens_seen:,} tokens')

    if step % 400 == 0 or step == STEPS:
        wsm.record_snapshot(model)
        consolidated = wsm.build_consolidated_state_dict(fallback_model=model)
        ckpt_path = vault.save_checkpoint(
            'latest',
            model,
            tok,
            metrics={'val_ce_loss': round(float(out.ce_loss), 4), 'step': step, 'total_tokens_trained': tokens_seen, 'colab_gpu_trained': True},
            consolidated_state_dict=consolidated,
        )
        print(f'💾 Checkpoint sauvegardé -> {ckpt_path}')

In [ ]:
# 5️⃣ TEST EN DIRECT DE NEVER-AI (AVEC MÉMOIRE TITANS CONTINUE)
from never_ai.core.special_tokens import BOS_TOKEN, EOS_ID, NEVER_TOKEN, USER_ID, USER_TOKEN

model.eval()
stop_ids = {EOS_ID, USER_ID}
titans_state = None

for msg in [
    'tu t\'appelles comment ?',
    'ça dit quoi aujourd\'hui ? tu penses quoi des gens sur X ?',
]:
    p_ids = tok.encode(f'{BOS_TOKEN}{USER_TOKEN}{msg}{NEVER_TOKEN}')
    gen_ids, titans_state = model.generate(p_ids, max_new_tokens=90, temperature=0.72, top_p=0.92, stop_token_ids=stop_ids, titans_states=titans_state)
    reply = tok.decode(gen_ids[len(p_ids):], skip_special_tokens=True).strip()
    print(f'👤 Vous     : {msg}')
    print(f'🤖 Never-AI : {reply}\n')

In [ ]:
# 6️⃣ 🌍 PUBLIER CET ENTRAÎNEMENT DANS LE MODÈLE MONDIAL NEVER-AI SUR GITHUB (0 Mo SUR VOTRE RÉSEAU !)
from getpass import getpass
import os, zipfile

print('🌍 Contribution au modèle officiel partagé mouusieur/never-ai sur GitHub Releases')
print('   -> Entrez un GitHub Personal Access Token (droit Contents: Read & Write) pour publier directement depuis Google Cloud (0 Mo sur votre connexion),')
print('   -> OU appuyez simplement sur Entrée pour télécharger le .zip sur votre appareil.')
gh_tok = getpass('🔑 GitHub Token (ou Entrée pour télécharger en local) : ').strip()

if gh_tok:
    res = sync.contribute_to_global_release(
        model=model,
        tokenizer=tok,
        vault=vault,
        local_tokens_trained=tokens_seen,
        contributor='colab-gpu-trainer',
        github_token=gh_tok,
    )
    print('✅ CERVEAU MONDIAL NEVER-AI MIS À JOUR SUR GITHUB !', res['lineage'])
    print('👉 Tous les appareils (PC, Android Cloud, prochains entraînements Colab) profitent maintenant de cet apprentissage !')
else:
    zip_out = '/content/never-ai-brain-sprout-moe.zip'
    latest_dir = os.path.join(VAULT_DIR, 'latest')
    with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in sorted(os.listdir(latest_dir)):
            fpath = os.path.join(latest_dir, fname)
            if os.path.isfile(fpath):
                zf.write(fpath, arcname=fname)
    print(f'📦 Archive locale prête : {zip_out} ({os.path.getsize(zip_out)/1048576:.1f} Mo)')
    from google.colab import files
    files.download(zip_out)